# Measuring PSF moments (`psf_fomo.moments`)

This notebook is a tutorial for the moments module: how to measure the size and shape of
simulated stars, how to choose the algorithms and images, how to compare simulations with
different PSF errors, and how to look at the results.

The module follows the same pattern as the simulations:

1. **create** a `MomentsValidation` with one or more simulations,
2. **configure** it: `set_algorithms`, `set_images`, `select`,
3. **run** it, which returns a `MomentsResult`: a table with one row per measurement,
   summaries and plots.

Single images can also be measured directly with `measure_star`.

Definitions (size `T`, shape `e1`, `e2`, units) are in
[`docs/conventions.md`](../docs/conventions.md#moments); all options are in
[`docs/moments.md`](../docs/moments.md). The algorithms other than `hsm` need `ngmix`.
Nothing in this notebook is written to disk; the cells that save are commented out.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import NullFormatter, ScalarFormatter
import galsim
import galsim.roman
from astropy.table import Table

from psf_fomo.sim import RomanSimulation
from psf_fomo.moments import MomentsValidation, measure_star
from psf_fomo.moments.measure import make_star

**Plot style.** One style for every figure, and one colour per algorithm, the same in every
plot. Change these lines to change the look of the whole notebook.

In [ ]:
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
COLORS = {"hsm": PALETTE[0], "gauss": PALETTE[1], "wmom_0.4": PALETTE[2]}
REFERENCE = MUTED                        # injected / reference values
STAR_CMAP = LinearSegmentedColormap.from_list("star", [SURFACE, "#86b6ef", "#2a78d6", "#0d366b"])

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.size": 12, "text.color": INK,
    "axes.titlesize": 13, "axes.titleweight": "semibold", "axes.titlelocation": "left",
    "axes.titlepad": 10, "axes.labelsize": 12, "axes.labelcolor": MUTED,
    "axes.edgecolor": "#b8b7b1", "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 11, "ytick.labelsize": 11,
    "legend.frameon": False, "legend.fontsize": 10,
    "lines.linewidth": 2, "lines.markersize": 7, "errorbar.capsize": 0,
    "axes.prop_cycle": cycler(color=PALETTE),
})

**Helpers.** `make_sim` builds a simulation of SCA 1 in H158 with the stars on a grid (each
at a random sub-pixel phase); `measure` runs a `MomentsValidation` with `hsm`, `gauss` and
`wmom`; `fmt` rounds the numbers of a table for display.

In [ ]:
ALGS = ["hsm", "gauss", "wmom_0.4"]      # algorithm names in the result tables
QUANTITIES = {"T": "T [pix²]", "e1": "e1", "e2": "e2"}


def make_sim(n_stars=25, seed=1):
    s = RomanSimulation(n_stars=n_stars, seed=seed)
    s.set_placement("grid")
    return s


def measure(sims, images=("noiseless", "oversampled")):
    mv = MomentsValidation(sims, reference="base")
    mv.set_algorithms(["hsm", "gauss", "wmom"], wmom_fwhm=0.4)
    mv.set_images(images, oversample=5)
    return mv.run()


def fmt(table):
    # 4 decimals; scientific notation below 1e-3.
    for name in table.colnames:
        if table[name].dtype.kind == "f":
            table[name].info.format = (
                lambda v: f"{v:.4f}" if v == 0 or 1e-3 <= abs(v) < 1e4 else f"{v:.3e}")
    return table


def good(table, **match):
    # Rows with flag 0 whose columns equal the given values.
    keep = table["flag"] == 0
    for column, value in match.items():
        keep &= table[column] == value
    return table[keep]


def one_to_one(ax, label="1:1"):
    lo = min(ax.get_xlim()[0], ax.get_ylim()[0])
    hi = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([lo, hi], [lo, hi], color=REFERENCE, lw=1, ls=":", label=label, zorder=1)
    ax.set(xlim=(lo, hi), ylim=(lo, hi))

## 1. Quick start: simulate, measure, look

A simulation of 49 stars on SCA 1 (magnitude 18, see `01_simulation.ipynb`), measured with
three algorithms on two kinds of image:

- **noiseless**: the star on the native 0.11″ pixels, re-drawn without noise;
- **oversampled**: the same star (pixel response included) on a grid 5 times finer, the
  reference for what the native measurement should give.

Noise is left out in most of this notebook: these stars are bright, and noiseless images
isolate the effect of the pixels. Section 7 adds noise.

In [ ]:
builder = RomanSimulation(n_stars=49, seed=1)
builder.set_placement("grid")
sim = builder.run()

mv = MomentsValidation({"base": sim})
mv.set_algorithms(["hsm", "gauss", "wmom"], wmom_fwhm=0.4)
mv.set_images(["noiseless", "oversampled"], oversample=5)
res = mv.run()

`res.table` has one row per simulation × star × image × algorithm: the star (`star_id`,
position `x`, `y`, sub-pixel phase), the measured size `T` [native pix²], shape `e1`, `e2`,
centroid `du`, `dv` [arcsec] and a `flag` (0 = the measurement worked).

In [ ]:
fmt(res.table["star_id", "x", "y", "image", "algorithm", "T", "e1", "e2", "du", "dv", "flag"][:6])

The measured moments of all stars: native noiseless stamps (top) and oversampled images
(bottom). `n` is the number of stars where the algorithm worked; failed measurements are left
out.

In [ ]:
def plot_moments(table, algorithms, images=("noiseless", "oversampled"), colors=COLORS):
    fig, axes = plt.subplots(len(images), 3, figsize=(15, 3.9 * len(images)), squeeze=False)
    for i, image in enumerate(images):
        rows = table[table["image"] == image]
        for j, (q, label) in enumerate(QUANTITIES.items()):
            ax = axes[i, j]
            values = {a: np.asarray(good(rows, algorithm=a)[q]) for a in algorithms}
            pooled = np.concatenate([v for v in values.values() if len(v)])
            bins = np.linspace(pooled.min(), pooled.max(), 20)
            for a, v in values.items():
                if len(v):
                    ax.hist(v, bins=bins, color=colors[a], alpha=0.25)
                    ax.hist(v, bins=bins, color=colors[a], histtype="step", lw=2,
                            label=f"{a}  (n = {len(v)})")
            ax.set(title=f"{image}: {q}", xlabel=label, ylabel="stars")
        axes[i, 0].legend()
    fig.tight_layout()
    plt.show()


plot_moments(res.table, ALGS)

`summary("moments")` gives the mean, the star-to-star standard deviation (`_std`) and the
error on the mean (`_err`) per image and algorithm, with the number of stars used (`n`) and
of failed measurements (`n_flagged`).

In [ ]:
fmt(res.summary("moments")["image", "algorithm", "n", "n_flagged", "T_mean", "T_std",
                           "e1_mean", "e2_mean"])

### Changing what is measured

- `set_algorithms(algorithms, wmom_fwhm)`: any of `"hsm"` (GalSim adaptive moments), `"am"`
  (ngmix adaptive moments), `"gauss"` (ngmix elliptical Gaussian fit) and `"wmom"` (moments
  with a fixed circular Gaussian weight of FWHM `wmom_fwhm` arcsec at the star position).
  Several FWHMs give one `wmom_<fwhm>` entry each.
- `set_images(images, oversample, include_pixel)`: any of `"noisy"` (the stamp from the
  simulated image), `"noiseless"` and `"oversampled"`; the oversampling factor; and whether
  the oversampled image includes the pixel response.
- `select(scas, obs)`: only some SCAs or observations.

`wmom` measures through a weight of fixed size, so its `T` is a different quantity for each
FWHM: compare `wmom` values only at the same FWHM. Below: the same stars with three weights,
oversampled images.

In [ ]:
mv = MomentsValidation({"base": sim})
mv.set_algorithms(["wmom"], wmom_fwhm=[0.3, 0.4, 0.6])
mv.set_images(["oversampled"])
res_fwhm = mv.run()

labels = ["wmom_0.3", "wmom_0.4", "wmom_0.6"]
plot_moments(res_fwhm.table, labels, images=["oversampled"],
             colors=dict(zip(labels, PALETTE)))

## 2. Measuring a single image

`measure_star(star, algorithm)` measures one Piff `Star`, and `make_star(image, position,
weight)` turns any GalSim image into one: a simulated stamp, a truth image, or an image you
drew yourself. The position is where the measurement starts (`wmom`: where its weight is
centred), in the image's pixel coordinates.

In [ ]:
stars, rows = sim.stars(sca=1)                   # Piff stars cut from the simulated image
star = stars[0]
noiseless = make_star(sim.render_stamp(0), star.image_pos, star.weight)
fine = sim.truth_psf(sca=1, obs=0, star_id=0, oversample=5)
oversampled = make_star(fine, fine.true_center)

gaussian = galsim.Gaussian(sigma=0.08).shear(e1=0.05).drawImage(nx=25, ny=25, scale=0.11)
own = make_star(gaussian, gaussian.true_center)  # any GalSim image

images = {"noiseless (native)": noiseless, "oversampled ×5": oversampled,
          "your own image (Gaussian)": own}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, s) in zip(axes, images.items()):
    a = s.image.array
    ax.imshow(np.log10(np.maximum(a / a.max(), 1e-4)), origin="lower", cmap=STAR_CMAP)
    ax.set_title(name)
    ax.axis("off")
fig.suptitle("log10(image / peak)", x=0.02, ha="left", color=MUTED, fontsize=11)
fig.tight_layout()
plt.show()

rows = []
for name, s in images.items():
    for alg in ("hsm", "am", "gauss", "wmom"):
        m = measure_star(s, alg, weight_fwhm=0.4)
        rows.append({"image": name, "algorithm": alg, "T [pix²]": m["T"], "e1": m["e1"],
                     "e2": m["e2"], "flag": m["flag"]})
fmt(Table(rows=rows))

## 3. Native pixels vs the oversampled image

The same star measured on the native pixels and on the oversampled image, with the same
algorithm: each point is one star of section 1. `res.bias()` returns these pairs per star
(`dT_T` = (T_native − T_oversampled) / T_oversampled, `de1`, `de2`), and `summary("bias")`
averages them.

In [ ]:
pairs = res.bias()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (q, label) in zip(axes, QUANTITIES.items()):
    for a in ALGS:
        r = good(pairs, algorithm=a)
        ax.scatter(r[q + "_os"], r[q], s=40, color=COLORS[a], edgecolor=SURFACE,
                   lw=0.8, label=f"{a}  (n = {len(r)})", zorder=3)
    one_to_one(ax)
    ax.set(title=q, xlabel=f"oversampled {label}", ylabel=f"native {label}")
axes[0].legend()
fig.tight_layout()
plt.show()

fmt(res.summary("bias")["algorithm", "n", "n_flagged", "dT_T_mean", "dT_T_std", "de1_mean",
                        "de2_mean"])

The result also has ready-made plots: `plot_bias` (the mean differences above),
`plot_recovery`, `plot_vs_phase` and `plot_vs_snr` (see `docs/moments.md`). For example:

In [ ]:
res.plot_bias(images="noiseless");

## 4. Comparing simulations: measured vs injected PSF errors

To see how well a PSF error is measured, compare a simulation with the error to one without.
Simulations with the same seed have the same stars (positions, magnitudes, noise), so they
are compared star by star. Give `MomentsValidation` several simulations and name the
reference:

`MomentsValidation({"base": ..., "defocus": ..., "shear": ...}, reference="base")`

`res.differences()` then gives, for every star, the **measured error** (each simulation −
the reference, same image and algorithm: `dT_T`, `de1`, `de2`) and the **injected error** of
the same star, measured with HSM on the oversampled images (`dT_T_hsm_os`, `de1_hsm_os`, ...).
The injected error is not the nominal value set in the simulation, because the errors act
on the optics before the pixel.

In [ ]:
defocus = make_sim()
defocus.set_zernikes({4: 0.05})                  # extra defocus, waves
shear = make_sim()
shear.add_shear(g1=0.01)                         # shear of the optical PSF
sims4 = {"base": make_sim().run(), "defocus": defocus.run(), "shear": shear.run()}
res4 = measure(sims4)

d = res4.summary("differences")
fmt(d[d["image"] == "noiseless"]["sim", "algorithm", "n", "dT_T_mean", "dT_T_hsm_os_mean",
                                 "de1_mean", "de1_hsm_os_mean"])

Per star: measured against injected error, native noiseless stamps. Left: the size change
from the defocus; right: `e1` from the shear.

In [ ]:
diff4 = res4.differences()
cases = [("defocus", "dT_T", "dT/T"), ("shear", "de1", "de1")]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, q, label) in zip(axes, cases):
    for a in ALGS:
        r = good(diff4, sim=name, image="noiseless", algorithm=a)
        ax.scatter(r[q + "_hsm_os"], r[q], s=40, color=COLORS[a], edgecolor=SURFACE, lw=0.8,
                   label=f"{a}  (n = {len(r)})", zorder=3)
    one_to_one(ax, "measured = injected")
    ax.set(title=f"{name}: {label}", xlabel=f"injected {label}", ylabel=f"measured {label}")
    ax.legend()
fig.tight_layout()
plt.show()

## 5. Response to the size of the error

Three scans, each against its own `base` simulation: a dilation of the optical PSF (0.98,
0.99, 1.01, 1.02) for the size, and a shear `g1` or `g2` of the optical PSF (±0.01, ±0.02)
for the shape. For each value: the mean measured error over the stars against the mean
injected error; bars are the star-to-star standard deviation. The grey points are HSM on the
oversampled images, which is the injected error itself; `gauss` and `wmom` are measured on
the native noiseless stamps.

In [ ]:
def response(add_error, values, n_stars=25):
    # base + one simulation per value, named "+0.01" etc.
    sims = {"base": make_sim(n_stars).run()}
    for v in values:
        s = make_sim(n_stars)
        add_error(s, v)
        sims[f"{v:+g}"] = s.run()
    return measure(sims)


VALUES = [-0.02, -0.01, 0.01, 0.02]
scans = {"dT_T": response(lambda s, v: s.add_dilation(1 + v), VALUES),
         "de1": response(lambda s, v: s.add_shear(g1=v), VALUES),
         "de2": response(lambda s, v: s.add_shear(g2=v), VALUES)}
TITLES = {"dT_T": "dilation → dT/T", "de1": "shear g1 → e1", "de2": "shear g2 → e2"}

In [ ]:
def mean_points(diff, q, algorithm, image):
    # (injected, measured mean, measured std) per simulation, sorted by injected value.
    rows = good(diff, algorithm=algorithm, image=image)
    points = [(np.mean(r[q + "_hsm_os"]), np.mean(r[q]), np.std(r[q]))
              for r in (rows[rows["sim"] == s] for s in set(rows["sim"]))]
    return np.array(sorted(points)).T


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (q, res_q) in zip(axes, scans.items()):
    diff = res_q.differences()
    x, y, s = mean_points(diff, q, "hsm", "oversampled")
    ax.errorbar(x, y, yerr=s, color=REFERENCE, marker="o", ls="--", lw=1.5,
                label="hsm, oversampled (= injected)")
    for a in ("gauss", "wmom_0.4"):
        x, y, s = mean_points(diff, q, a, "noiseless")
        ax.errorbar(x, y, yerr=s, color=COLORS[a], marker="o", label=f"{a}, native")
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.axvline(0, color=MUTED, lw=0.8)
    ax.set(title=TITLES[q], xlabel=f"injected {q}", ylabel=f"measured {q}")
axes[0].legend()
fig.tight_layout()
plt.show()

`hsm` is not shown on the native stamps because it fails on many of them. Number of native
noiseless measurements that failed, per scan (all five simulations of the scan):

In [ ]:
counts = []
for q, res_q in scans.items():
    t = res_q.table[res_q.table["image"] == "noiseless"]
    row = {"scan": TITLES[q], "measurements": int(np.sum(t["algorithm"] == "hsm"))}
    for a in ALGS:
        row[f"{a} failed"] = int(np.sum((t["algorithm"] == a) & (t["flag"] != 0)))
    counts.append(row)
Table(rows=counts)

## 6. Sub-pixel phase

Where the star centre falls within a pixel (its sub-pixel phase) changes the native image,
and so the measurement. Each point is one star of the scans in section 5: measured / injected
error against the distance of the star centre from the pixel centre (0 = pixel centre,
0.71 = pixel corner), native noiseless stamps. Points outside the plotted range are not shown.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
for ax, (q, res_q) in zip(axes, scans.items()):
    diff = res_q.differences()
    for a in ALGS:
        r = good(diff, image="noiseless", algorithm=a)
        ax.scatter(np.hypot(r["phase_x"], r["phase_y"]), r[q] / r[q + "_hsm_os"], s=30,
                   color=COLORS[a], edgecolor=SURFACE, lw=0.6, alpha=0.9,
                   label=f"{a}  (n = {len(r)})", zorder=3)
    ax.axhline(1, color=REFERENCE, lw=1, ls=":")
    ax.set(ylim=(-0.5, 2.0), xlim=(0, 0.72), title=TITLES[q],
           xlabel="distance from the pixel centre [pix]")
    ax.legend(loc="lower left")
axes[0].set_ylabel("measured / injected (per star)")
fig.tight_layout()
plt.show()

## 7. Centroid errors

Every measurement starts from a position: the catalog position here, the fitted star centre
in a PSF fit. What the position does depends on the algorithm:

- `wmom` puts its weight at that position and does not move it, so a wrong position changes
  the measurement directly;
- `hsm` and `gauss` fit their own centroid; the position is only their starting guess.

The tests below measure the noiseless stamps of section 4 about deliberately wrong positions
(the pixels are unchanged; only the position given to `make_star` moves). For `hsm` and
`gauss` they test a wrong starting guess. A wrong fitted centre in a PSF fit is a different
case: the model is then drawn at that centre, which changes its sub-pixel phase (section 6).

In [ ]:
def noiseless_stars(sim):
    stars, rows = sim.stars(sca=1)
    return [make_star(sim.render_stamp(int(r["star_id"])), s.image_pos, s.weight)
            for s, r in zip(stars, rows)], np.array(rows["star_id"])


def measure_shifted(stars, offsets, algorithm):
    # Same pixels; measurement position moved by (dx, dy) native pixels.
    return Table(rows=[measure_star(make_star(s.image, s.image_pos + galsim.PositionD(dx, dy),
                                              s.weight), algorithm, weight_fwhm=0.4)
                       for s, (dx, dy) in zip(stars, offsets)])


def mean(values):
    return np.mean(values) if len(values) else np.nan


CEN_ALGS = {"hsm": "hsm", "gauss": "gauss", "wmom_0.4": "wmom"}   # label → algorithm
base_stars, star_ids = noiseless_stars(sims4["base"])
defocus_stars, _ = noiseless_stars(sims4["defocus"])
n = len(base_stars)

**A shifted position.** Every star measured about a position shifted by `s` pixels along x.
Left and middle: the change in `T` and `e1` relative to the measurement at the true position.
Right: the centroid each algorithm reports, relative to the true position (dotted: the
position it was given).

In [ ]:
SHIFTS = [0.0, 0.05, 0.1, 0.2, 0.3]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for label, alg in CEN_ALGS.items():
    at_truth = measure_shifted(base_stars, np.zeros((n, 2)), alg)
    dT, de1, cen = [], [], []
    for shift in SHIFTS:
        m = measure_shifted(base_stars, np.tile([shift, 0.0], (n, 1)), alg)
        ok = (m["flag"] == 0) & (at_truth["flag"] == 0)
        dT.append(mean(m["T"][ok] / at_truth["T"][ok] - 1))
        de1.append(mean(m["e1"][ok] - at_truth["e1"][ok]))
        cen.append(mean(m["du"][ok] / galsim.roman.pixel_scale + shift))  # du: from given position
    for ax, values in zip(axes, (dT, de1, cen)):
        ax.plot(SHIFTS, values, "o-", color=COLORS[label], label=label)
axes[2].plot(SHIFTS, SHIFTS, color=REFERENCE, ls=":", lw=1, label="given position")
axes[0].set(title="size", ylabel="T / T(true position) − 1")
axes[1].set(title="shape", ylabel="e1 − e1(true position)")
axes[2].set(title="reported centroid", ylabel="centroid − true position [pix]")
for ax in axes:
    ax.set_xlabel("position shift along x [pix]")
axes[0].legend()
axes[2].legend()
fig.tight_layout()
plt.show()

**Measured PSF errors with wrong positions.** The size change from the defocus, measured with
a random position error (Gaussian, rms per axis on the x axis) in two ways:

- **same** (solid): a star's base and defocused images are measured about the same wrong
  position, as when data and model of one star are compared about one common centre;
- **independent** (dashed): each image gets its own position error.

Left: measured `dT/T`, mean over stars ± error on the mean; dotted: the mean injected value.
Right: star-to-star standard deviation of measured − injected `dT/T` (logarithmic axis).

In [ ]:
inj = good(res4.differences(), sim="defocus", image="oversampled", algorithm="hsm")
injected = dict(zip(inj["star_id"], inj["dT_T_hsm_os"]))
injected = np.array([injected[i] for i in star_ids])

RMS = [0.0, 0.05, 0.1, 0.2]
rng = np.random.default_rng(0)
cen_rows = []
for rms in RMS:
    offsets_base = rng.normal(0.0, rms, (n, 2))
    offsets_other = rng.normal(0.0, rms, (n, 2))
    for label, alg in CEN_ALGS.items():
        mb = measure_shifted(base_stars, offsets_base, alg)
        for scenario, offsets in (("same", offsets_base), ("independent", offsets_other)):
            mo = measure_shifted(defocus_stars, offsets, alg)
            ok = (mb["flag"] == 0) & (mo["flag"] == 0)
            measured = np.asarray(mo["T"][ok] / mb["T"][ok] - 1)
            cen_rows.append({"rms": rms, "algorithm": label, "scenario": scenario,
                             "mean": mean(measured),
                             "err": measured.std() / np.sqrt(len(measured)) if len(measured) else np.nan,
                             "std": (measured - injected[ok]).std() if len(measured) else np.nan})
cen_table = Table(rows=cen_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for k, label in enumerate(CEN_ALGS):
    for scenario, ls in (("same", "-"), ("independent", "--")):
        t = cen_table[(cen_table["algorithm"] == label) & (cen_table["scenario"] == scenario)]
        x = np.asarray(t["rms"]) + 0.003 * (k - 1)
        axes[0].errorbar(x, t["mean"], yerr=t["err"], color=COLORS[label], ls=ls, marker="o",
                         label=f"{label}, {scenario}")
        axes[1].plot(x, t["std"], color=COLORS[label], ls=ls, marker="o")
axes[0].axhline(injected.mean(), color=REFERENCE, ls=":", lw=1, label="injected (mean)")
axes[0].set(title="mean size change", ylabel="measured dT/T")
axes[1].set(title="star-to-star scatter", ylabel="std of measured − injected dT/T")
axes[1].set_yscale("log")
for ax in axes:
    ax.set_xlabel("position error rms per axis [pix]")
axes[0].legend(ncol=2, fontsize=9)
fig.tight_layout()
plt.show()

## 8. Noise and signal-to-noise

Now with noise: stars from magnitude 18 to 23 with the Roman noise and detector effects
(`set_noise("roman")`), a base simulation and one with the defocus of section 4, measured on
the **noisy** stamps (`set_images(["noisy", "oversampled"])`). The S/N of a star is
√(Σ I² / σ²) over its noiseless stamp, σ² the background variance (column `snr`). Stars are
grouped in five S/N bins with the same number of stars each:

1. native − oversampled `dT/T` of the base stars (mean ± error on the mean);
2. measured `dT/T` of the defocus (mean ± error on the mean); dotted: the mean injected value;
3. star-to-star standard deviation of measured − injected `dT/T`.

`hsm` is left out: it fails on many native stamps (section 5).

In [ ]:
def faint_sim(defocus=0.0):
    s = make_sim(n_stars=60, seed=3)
    s.set_brightness(mag_range=(18, 23))
    s.set_noise("roman")
    s.set_zernikes({4: defocus})
    return s.run()


res_snr = measure({"base": faint_sim(), "defocus": faint_sim(0.05)},
                  images=("noisy", "oversampled"))


def snr_bins(snr, values, n_bins=5):
    # Equal-count S/N bins: median S/N, mean, error on the mean, std.
    edges = np.quantile(snr, np.linspace(0, 1, n_bins + 1))
    idx = np.clip(np.digitize(snr, edges[1:-1]), 0, n_bins - 1)
    out = []
    for b in range(n_bins):
        v = values[idx == b]
        out.append((np.median(snr[idx == b]), v.mean(), v.std() / np.sqrt(len(v)), v.std()))
    return np.array(out).T


bias = good(res_snr.bias(), sim="base", image="noisy")
diff = good(res_snr.differences(), image="noisy")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for a in ("gauss", "wmom_0.4"):
    b = bias[bias["algorithm"] == a]
    x, m, e, _ = snr_bins(np.asarray(b["snr"]), np.asarray(b["dT_T"]))
    axes[0].errorbar(x, m, yerr=e, color=COLORS[a], marker="o", label=a)
    d = diff[diff["algorithm"] == a]
    x, m, e, _ = snr_bins(np.asarray(d["snr"]), np.asarray(d["dT_T"]))
    axes[1].errorbar(x, m, yerr=e, color=COLORS[a], marker="o", label=a)
    x, _, _, s = snr_bins(np.asarray(d["snr"]), np.asarray(d["dT_T"] - d["dT_T_hsm_os"]))
    axes[2].plot(x, s, color=COLORS[a], marker="o", label=a)
axes[1].axhline(np.mean(diff["dT_T_hsm_os"]), color=REFERENCE, ls=":", lw=1,
                label="injected (mean)")
axes[0].set(title="native − oversampled (base)", ylabel="dT/T")
axes[1].set(title="measured size change (defocus)", ylabel="dT/T")
axes[2].set(title="star-to-star scatter (defocus)", ylabel="std of measured − injected dT/T")
axes[2].set_yscale("log")
for ax in axes:
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel("S/N")
    ax.legend()
fig.tight_layout()
plt.show()

## 9. Saving

A result can be saved to a folder (`moments.fits` with the table, `config.yaml` with the
configuration and software versions) and reloaded. A `MomentsValidation` whose simulations
were given as saved folders can also be written to YAML and rebuilt. The cells are commented
out so that running the notebook writes nothing.

In [ ]:
# Save while running, or afterwards, and reload:
# res = mv.run(save_to="out/moments01")
# res.save("out/moments01")
# from psf_fomo.moments import load_moments
# res = load_moments("out/moments01")

# A configuration file, with the simulations given as saved folders:
# mv = MomentsValidation({"base": "out/sim_base", "defocus": "out/sim_defocus"}, reference="base")
# mv.to_yaml("moments.yaml")
# res = MomentsValidation.from_yaml("moments.yaml").run()

## Next

- `03_piff_validation.ipynb`: fitting the simulations with Piff and measuring the fitted PSF
  model with the same algorithms.